# 01. OpenVINO 첫걸음 — 얼굴 찾기 모델을 CPU · GPU · NPU 에서

> 모든 단원의 뼈대가 되는 흐름을 익힌다: **모델 열기 → 장치에 맞게 컴파일 → 전처리 → 추론 → 후처리**.
> 그리고 같은 모델을 장치마다 돌려 속도를 제대로 재는 법을 배운다.

### 이 시간에 배우는 것
1. OpenVINO 모델 파일(IR: `.xml` + `.bin`)과 `Core` · `read_model` · `compile_model`
2. 모델 설명서 읽기: 입력 · 출력의 이름과 모양
3. 전처리(사진 → 모델 입력)와 후처리(모델 출력 → 상자)
4. 속도 재기: 컴파일 시간 · 첫 추론 · 평균 · 처리량(비동기)

### 전체 흐름
```
사진(BGR, 480x640)  ──전처리──▶  [1,3,300,300] float32  ──추론(CPU/GPU/NPU)──▶  [1,1,N,7]  ──후처리──▶  얼굴 상자들
```

## 1. 준비 — `Core` 는 OpenVINO 의 입구
`ov.Core()` 하나가 이 PC 의 장치들을 찾아 두고, 모델을 읽고 컴파일하는 일을 맡는다.

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

for d in core.available_devices:
    print(f"{d:6s}", core.get_property(d, "FULL_DEVICE_NAME"))

## 2. 모델 열기 — `read_model`
OpenVINO 모델(IR)은 두 파일이다.
- `.xml` — 구조: 어떤 계산(연산)이 어떤 순서로 이어지는지
- `.bin` — 가중치: 학습으로 정해진 숫자들

`read_model` 은 파일을 읽어 **장치와 상관없는** 모델 객체를 만든다. 아직 어느 장치에서도 돌릴 수 없다.

사용하는 모델: **face-detection-retail-0005** (인텔 Open Model Zoo). 매장 카메라처럼 정면 · 중간 거리 얼굴에 맞춰 학습됐다.

In [ ]:
MODEL = MODELS / "face" / "face-detection-retail-0005.xml"
assert MODEL.exists(), f"모델이 없어요: {MODEL}"

model = core.read_model(MODEL)
inp, out = model.input(0), model.output(0)
print("입력:", inp.any_name, inp.partial_shape, inp.element_type)
print("출력:", out.any_name, out.partial_shape, out.element_type)
print("연산 수:", len(model.get_ops()))

**모델 설명서 읽기**
- 입력 `[1, 3, 300, 300]` — **NCHW**: 사진 1장 · 색 3채널(**B, G, R** 순서) · 세로 300 · 가로 300. 값은 0~255 그대로.
- 출력 `[1, 1, N, 7]` — 후보 N개(이 모델은 200개), 후보마다 숫자 7개:

| 자리 | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| 뜻 | 이미지 번호 | 종류(1=얼굴) | 점수 0~1 | x1 | y1 | x2 | y2 |

  좌표는 **0~1 비율**이다 (0 = 왼쪽/위 끝, 1 = 오른쪽/아래 끝). 사진 크기를 곱해야 픽셀이 된다.
  이미지 번호가 **-1** 이면 거기서 목록이 끝난다.
- `element_type` `f32` — float32 를 넣어야 한다.

이 정보는 모델마다 다르다. 실무에서는 모델 설명서(model card)를 먼저 찾아 읽고, 모르면 위처럼 직접 찍어 본다.

## 3. 사진 한 장

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 4. 전처리 — 사진을 입력 모양으로
00 번 6절의 다섯 가지 중 ② 크기, ⑤ 축 순서, ④ 숫자 종류만 쓴다. 이 모델은 BGR 로 배웠고 0~255 를 받으므로 색 순서 · 숫자 범위는 그대로 둔다.

In [ ]:
def preprocess(bgr):
    x = cv2.resize(bgr, (300, 300))     # ① 크기: 300 x 300 (비율이 깨져도 이 모델은 그렇게 배웠다)
    x = x.transpose(2, 0, 1)            # ② 순서: (세로, 가로, 채널) → (채널, 세로, 가로)
    x = x[None].astype(np.float32)      # ③ 맨 앞에 '사진 몇 장' 차원 + float32
    return x

blob = preprocess(frame)
print(frame.shape, frame.dtype, "→", blob.shape, blob.dtype)

## 5. 컴파일 · 추론 — `compile_model`
`compile_model(model, "CPU")` 는 모델을 **그 장치에서 가장 빠르게 돌도록** 바꾼다 (연산 합치기, 메모리 배치, 장치용 코드 생성).
결과인 `CompiledModel` 은 함수처럼 부를 수 있다: `compiled(입력)` → 출력 사전. 출력은 `compiled.output(0)` 같은 **포트**로 꺼낸다.

In [ ]:
compiled = core.compile_model(model, "CPU")
res = compiled(blob)                         # 추론 한 번
dets = res[compiled.output(0)].reshape(-1, 7)  # [1,1,N,7] → [N,7] 로 펴서 다루기 쉽게
print("후보 수:", len(dets))
print("앞의 5개 (번호, 종류, 점수, x1, y1, x2, y2):")
print(np.round(dets[:5], 3))

## 6. 후처리 — 숫자를 얼굴 상자로
1. 이미지 번호가 -1 이면 멈춘다 (목록 끝).
2. 점수가 기준(`conf`)보다 낮으면 버린다.
3. 0~1 좌표에 사진의 가로 · 세로를 곱해 픽셀로 바꾼다.

**점수 기준의 뜻**: 낮추면 놓치는 얼굴이 줄지만 엉뚱한 것도 얼굴로 잡는다. 높이면 반대. 쓰임새에 따라 정한다.

In [ ]:
def faces(dets, w, h, conf=0.5):
    boxes = []
    for image_id, label, score, x1, y1, x2, y2 in dets:
        if image_id < 0:                      # -1 = 후보 목록 끝
            break
        if score < conf:
            continue
        boxes.append((int(x1 * w), int(y1 * h), int(x2 * w), int(y2 * h), float(score)))
    return boxes

h, w = frame.shape[:2]
found = faces(dets, w, h)
view = frame.copy()                           # 원본은 두고 복사본에 그린다
for x1, y1, x2, y2, s in found:
    cv2.rectangle(view, (x1, y1), (x2, y2), (0, 200, 0), 3)
    cv2.putText(view, f"{s:.2f}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2)
print("찾은 얼굴:", len(found), found)
show(view)

## 7. 속도 재기 — 제대로 재는 법
처음 한 번은 느리다 (메모리 준비, 장치 초기화). 그래서 세 가지를 따로 잰다.
- **컴파일**: 모델을 그 장치용으로 바꾸는 시간. NPU · GPU 는 처음에 몇 초 걸릴 수 있다 → 8절의 캐시로 줄인다.
- **첫 추론**: 컴파일 직후 한 번
- **평균**: 몇 번 예열한 뒤 100번 돌린 평균 — 실제 속도는 이것으로 말한다

`create_infer_request()` 는 추론 한 건을 담는 그릇이다. 매번 새로 만들지 않고 재사용하면 더 빠르다.

In [ ]:
def bench(device, n=100):
    t0 = time.perf_counter()
    c = core.compile_model(model, device)
    compile_s = time.perf_counter() - t0

    req = c.create_infer_request()
    t0 = time.perf_counter()
    req.infer({0: blob})
    first_ms = (time.perf_counter() - t0) * 1000

    for _ in range(5):                        # 예열
        req.infer({0: blob})
    t0 = time.perf_counter()
    for _ in range(n):
        req.infer({0: blob})
    avg_ms = (time.perf_counter() - t0) / n * 1000
    return compile_s, first_ms, avg_ms

rows = []
for dev in core.available_devices:
    try:
        rows.append((dev, *bench(dev)))
    except Exception as ex:
        print(dev, "실패:", ex)

print(f"{'장치':6s}{'컴파일(s)':>10s}{'첫 추론(ms)':>12s}{'평균(ms)':>10s}{'FPS':>8s}")
for dev, cs, fm, am in rows:
    print(f"{dev:6s}{cs:10.2f}{fm:12.1f}{am:10.2f}{1000 / am:8.0f}")

In [ ]:
# 막대로 보기 — 짧을수록 빠르다
best = min(r[3] for r in rows)
for dev, cs, fm, am in rows:
    print(f"{dev:6s} {'█' * max(1, round(am / best * 10))} {am:.2f} ms")

**읽는 법** — 작은 모델은 CPU 가 가장 빠르게 나오기도 한다. GPU · NPU 는 장치로 숫자를 보내고 받는 비용이 있어서,
계산이 작으면 그 비용이 더 크다. 대신 NPU 는 **전기를 적게** 쓰고 CPU 를 비워 둔다 (다른 일을 동시에 하기 좋다).
"어느 장치가 빠른가" 는 모델 크기에 따라 다르다 — 02 번(YOLO, 큰 모델)에서 다시 비교해 보자.

## 8. 컴파일 캐시 — 두 번째부터 빨리
`CACHE_DIR` 을 정해 두면 컴파일 결과를 디스크에 저장하고, 다음에는 읽기만 한다. NPU · GPU 에서 효과가 크다.

In [ ]:
core.set_property({"CACHE_DIR": "ov-cache"})
dev = "NPU" if "NPU" in core.available_devices else ("GPU" if "GPU" in core.available_devices else "CPU")
for i in range(2):
    t0 = time.perf_counter()
    core.compile_model(model, dev)
    print(f"{dev} 컴파일 {i + 1}번째: {(time.perf_counter() - t0) * 1000:.0f} ms")

## 9. 한 장씩 vs 여러 장 동시에 — 비동기 추론
지금까지는 **한 장 넣고 → 기다리고 → 다음 장** (동기). 영상처럼 장수가 많으면 여러 장을 **동시에** 넣어
장치를 쉬지 않게 하는 편이 초당 처리량(FPS)이 높다. `AsyncInferQueue` 가 그 일을 한다.
- **지연(latency)**: 한 장이 나올 때까지 걸리는 시간 — 반응 속도
- **처리량(throughput)**: 1초에 몇 장 — 영상 처리량

In [ ]:
c = core.compile_model(model, "CPU", {"PERFORMANCE_HINT": "THROUGHPUT"})   # 처리량 위주로 컴파일
n_req = c.get_property("OPTIMAL_NUMBER_OF_INFER_REQUESTS")
queue = ov.AsyncInferQueue(c, n_req)
done = []
queue.set_callback(lambda request, userdata: done.append(userdata))       # 한 장 끝날 때마다 불린다

N = 200
t0 = time.perf_counter()
for i in range(N):
    queue.start_async({0: blob}, userdata=i)   # 기다리지 않고 계속 넣는다
queue.wait_all()
fps_async = N / (time.perf_counter() - t0)
print(f"동시 요청 {n_req}개 · {len(done)}장 · {fps_async:.0f} FPS (동기 평균은 위 표의 CPU FPS)")

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| 얼굴을 하나도 못 찾음 | 너무 멀거나 옆얼굴 · 어두움 → `conf` 를 낮춰 보거나 가까이. 전처리 순서(축 바꾸기)를 빠뜨려도 이렇다 |
| `Exception ... shape` | 입력 모양이 다름 → `blob.shape` 가 `(1, 3, 300, 300)` 인지 |
| NPU 컴파일 실패 | 드라이버 · 지원 안 되는 연산 → 그 장치는 건너뛴다 (`bench` 가 "실패" 로 찍는다) |
| 상자가 엉뚱한 곳 | 가로 · 세로를 바꿔 곱했다 → x 에는 `w`, y 에는 `h` |

## 정리
- OpenVINO 흐름: `Core` → `read_model`(장치 무관) → `compile_model`(장치용) → `compiled(입력)` → 출력 포트로 꺼내기.
- 전처리는 **모델 설명서**대로: 크기 · 색 순서 · 숫자 범위 · 축 순서.
- 속도는 예열 뒤 평균으로. 지연과 처리량은 다르다. 캐시로 컴파일을 줄인다.

## 해 볼 것
1. `faces(dets, w, h, conf=0.3)` 과 `conf=0.8` 로 바꿔 보자. 무엇이 달라지나?
2. 장치를 `"AUTO"` 로 컴파일하고 무엇을 골랐는지 보자.
   <details><summary>힌트</summary>

   `c = core.compile_model(model, "AUTO"); print(c.get_property("EXECUTION_DEVICES"))`
   </details>
3. 9절을 `"NPU"` · `"GPU"` 로 바꿔 처리량을 비교해 보자.
4. 웹캠 영상으로 1초에 몇 번 얼굴을 찾는지 세어 보자 (`cv2.VideoCapture` 를 열어 두고 100장 반복).
   <details><summary>힌트</summary>

   ```python
   cap = cv2.VideoCapture(0); t0 = time.perf_counter()
   for _ in range(100):
       ok, f = cap.read(); faces(compiled(preprocess(f))[compiled.output(0)].reshape(-1, 7), f.shape[1], f.shape[0])
   cap.release(); print(100 / (time.perf_counter() - t0), "FPS")
   ```
   </details>

---
# 🛠 미니 프로젝트 — 얼굴 모자이크 카메라

**만들 것**: 웹캠 영상에서 얼굴을 찾아 **모자이크로 가리는** 실시간 카메라 (개인정보 보호). 화면에 FPS 와 얼굴 수를 띄운다.
15초 동안 돌고, 마지막 장면을 파일로 저장한다.

**쓰는 것**: `preprocess` · `faces` (이 노트북), 00 번의 모자이크 방법, `live` (아래 — 영상을 노트북 안에 갈아 끼워 보여 주는 함수)

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """웹캠 영상을 seconds 초 동안 process(사진) → 사진 으로 바꿔 노트북 안에 보여 준다. 멈추려면 I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # 웹캠이 없으면 사진으로
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n}장 · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (웹캠이 없어 사진으로 흉내)" if still is not None else ""))
    return out

In [ ]:
best_dev = min(rows, key=lambda r: r[3])[0] if rows else "CPU"    # 7절에서 가장 빨랐던 장치
fast = core.compile_model(model, best_dev)
last = [time.perf_counter()]

def mosaic_faces(f):
    h, w = f.shape[:2]
    found = faces(fast(preprocess(f))[fast.output(0)].reshape(-1, 7), w, h, conf=0.5)
    for x1, y1, x2, y2, s in found:
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
        roi = f[y1:y2, x1:x2]
        if roi.size:
            small = cv2.resize(roi, (max(1, roi.shape[1] // 12), max(1, roi.shape[0] // 12)))
            f[y1:y2, x1:x2] = cv2.resize(small, (roi.shape[1], roi.shape[0]), interpolation=cv2.INTER_NEAREST)
    now = time.perf_counter(); fps = 1 / max(now - last[0], 1e-6); last[0] = now
    cv2.putText(f, f"{best_dev} faces {len(found)}  {fps:.0f} FPS", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return f

final = live(mosaic_faces, seconds=15)
if final is not None:
    cv2.imwrite("privacy_cam.jpg", final); print("저장: privacy_cam.jpg")

**더 해 보기**
- 모자이크 대신 흐리기(`cv2.GaussianBlur`) · 검은 상자 · 이모지 그림 붙이기
- 얼굴이 **한 명보다 많을 때만** 가리기 (예: 수업 녹화에서 선생님 한 명은 보이게)
- `best_dev` 를 바꿔 FPS 비교